In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Create a variant analyses pipleline

Visit this [Github repository](https://github.com/Chisomgold/HackBio22-Team-Crick/tree/main/StageTwoAnalysis). It has a bash script and a link to a tutorial on Galaxy. Replicate the bash script using the same data but write your script in Python.

I.E., your final output should be a Python script that performs the same analyses in the bash script. It should be better than the bash script, esp in terms of file handling.



In [ ]:
# download data from zenodo
import os
import urllib.request

def download_data(url, raw_data):
  print(f'downloading data from zenodo...')
  # Create the output directory if it doesn't exist
  os.makedirs(raw_data, exist_ok=True)

  # list of files to download
  files = [
    "SLGFSK-N_231335_r1_chr5_12_17.fastq.gz",
    "SLGFSK-N_231335_r2_chr5_12_17.fastq.gz",
    "SLGFSK-T_231336_r1_chr5_12_17.fastq.gz",
    "SLGFSK-T_231336_r2_chr5_12_17.fastq.gz",
    "hg19.chr5_12_17.fa.gz"
    ]

  # download each  file
  for file in files:
    file_url = "https://zenodo.org/record/2582555/files/" + file
    file_path = os.path.join(raw_data, file) # Construct the full path
    print(f"Downloading {file}...")
    urllib.request.urlretrieve(file_url, file_path) # Download to the full path
    print(f"Done with {file}")

  print('All files downloaded successfully!')

download_data(url="https://zenodo.org/record/2582555", raw_data="raw_data")

In [4]:
!cp -r '/content/drive/MyDrive/project' '/content'

In [ ]:
#write the name of the files in raw_data folder into list.txt
file_names = ["SLGFSK-N_231335", "SLGFSK-T_231336"]
with open('list.txt', 'w') as f:
  for name in file_names:
    f.write(f'{name} \n')

print(f"File names written to list.txt")

File names written to list.txt


In [ ]:
import os

# installing packages (fastqc, multiqc, )
print("installing FastQC and MultiQC packages")
os.system('conda install -c bioconda fastqc multiqc -yes')

print("installation complete!")
print("Testing FastQC...")

installing FastQC and MultiQC packages
installation complete!
Testing FastQC...
/bin/bash: line 1: fastqc: command not found
Testing MultiQC...
/bin/bash: line 1: multiqc: command not found


In [ ]:
import os

print("Installing bioinformatics tools...")

os.system("apt-get update")
os.system("apt-get install -y fastqc multiqc")
# os.system("conda install -c bioconda trimmomatic --yes")
# !conda install -c bioconda trimmomatic --yes

print("Installation complete!")
! fastqc --version
!multiqc --version
# !trimmomatic --version

Installing bioinformatics tools...
Installation complete!
FastQC v0.11.9
multiqc, version 1.12


In [ ]:
import os
import subprocess
import glob

print("data processing...")

# create Fastqc_report directory
os.makedirs("/content/project/Fastqc_report", exist_ok=True)
print("Fastqc_Report directory created")

# Get a list of fastq.gz files in the raw_data directory
raw_data_path = "/content/project/raw_data"
fastq_files = glob.glob(os.path.join(raw_data_path, "*.fastq.gz"))

# process each fastq file
for fastq_file in fastq_files:
  try:
    subprocess.run(['fastqc', fastq_file, '-o', '/content/project/Fastqc_report'], check=True)
    print(f"FastQC report generated for {fastq_file}")
  except FileNotFoundError as e:
    print(f'Error running FastQC on {fastq_file}: {e}')
    print(f'Skipping file: {fastq_file}')
  except subprocess.CalledProcessError as e:
    print(f'Error running FastQC on {fastq_file}: {e}')
    print(f'Skipping file: {fastq_file}')

print("FastQC processing completed!")

data processing...
Fastqc_Report directory created
FastQC is installed and in PATH.
FastQC report generated for /content/project/raw_data/SLGFSK-N_231335_r2_chr5_12_17.fastq.gz
FastQC report generated for /content/project/raw_data/SLGFSK-T_231336_r2_chr5_12_17.fastq.gz
FastQC report generated for /content/project/raw_data/SLGFSK-N_231335_r1_chr5_12_17.fastq.gz
FastQC report generated for /content/project/raw_data/SLGFSK-T_231336_r1_chr5_12_17.fastq.gz
FastQC processing completed!


In [ ]:
import os
import subprocess
print("MultiQC processing...")

subprocess.run(['multiqc', '/content/project/Fastqc_report', '-o', '/content/project/Fastqc_report'])

MultiQC processing...


CompletedProcess(args=['multiqc', '/content/project/Fastqc_report', '-o', '/content/project/Fastqc_report'], returncode=0)

In [ ]:
import os
# !java -version
# download and unzip trimmomatic
# !wget http://www.usadellab.org/cms/uploads/supplementary/Trimmomatic/Trimmomatic-0.39.zip

print("Installing bioinformatics tools...")
# os.system("apt-get update")
# os.system("apt-get install -y fastqc multiqc")
# # os.system("conda install -c bioconda trimmomatic --yes")
# !conda install -c bioconda trimmomatic --yes

print("Installation complete!")
os.system("java -jar /content/Trimmomatic/Trimmomatic-0.39/trimmomatic-0.39.jar -version")
os.system("unzip Trimmomatic-0.39.zip -d /content/Trimmomatic")
# !java -jar /content/Trimmomatic/Trimmomatic-0.39/trimmomatic-0.39.jar -version
# !fastqc --version
# !multiqc --version


FastQC v0.11.9
multiqc, version 1.12


In [ ]:
import os
import subprocess
import glob

# create trimmed_reads and Fastqc_results directories
os.makedirs("/content/project/trimmed_reads", exist_ok=True)
os.makedirs("/content/project/trimmed_reads/Fastqc_results", exist_ok=True)

# get a list of forward read files in the raw_data directory
raw_data_path = "/content/project/raw_data"
r1_files = glob.glob(os.path.join(raw_data_path, "*_r1_*.fastq.gz"))

# process each pair of read files
for r1_path in r1_files:
  # extract the sample name from the r1 filename
  # assumes filename format is sample_name_r1_...fastq.gz
  sample = os.path.basename(r1_path).split('_r1_')[0]

  r2_path = os.path.join(raw_data_path, f"{sample}_r2_chr5_12_17.fastq.gz")

  # check if both paired files exist
  if not os.path.exists(r1_path):
    print(f"Warning: Forward read file not found: {r1_path}. Skipping sample {sample}.")
    continue
  if not os.path.exists(r2_path):
    print(f"Warning: Reverse read file not found: {r2_path}. Skipping sample {sample}.")
    continue

print(f"Processing sample: {sample}")

  # Define output paths for trimmed files
r1_paired_out = f'/content/project/trimmed_reads/{sample}_r1_paired.fq.gz'
r1_unpaired_out = f'/content/project/trimmed_reads/{sample}_r1_unpaired.fq.gz'
r2_paired_out = f'/content/project/trimmed_reads/{sample}_r2_paired.fq.gz'
r2_unpaired_out = f'/content/project/trimmed_reads/{sample}_r2_unpaired.fq.gz'

# Construct the Trimmomatic command using the java -jar command
trimmomatic_command = [
  'java', '-jar', '/content/Trimmomatic/Trimmomatic-0.39/trimmomatic-0.39.jar', 'PE', '-threads', '8',
  r1_path,
  r2_path,
  r1_paired_out,
  r1_unpaired_out,
  r2_paired_out,
  r2_unpaired_out,
  'ILLUMINACLIP:/content/Trimmomatic/Trimmomatic-0.39/adapters/TruSeq3-PE.fa:2:30:10:8:keepBothReads', # Assuming adapter file path
  'LEADING:3', 'TRAILING:10', 'MINLEN:25'
]

try:
  # run the Trimmomatic command
  print(f"Running Trimmomatic for {sample}...")
  subprocess.run(trimmomatic_command, check=True)
  print(f"Trimmomatic completed for {sample}")

  # construct the FastQC command 
  fastqc_cmd = [
    'fastqc',
    r1_paired_out,
    r2_paired_out,
    '-o', '/content/project/trimmed_reads/Fastqc_results'
  ]

  # run the FastQC command on trimmed paired reads
  print(f"Running FastQC on trimmed reads for {sample}...")
  subprocess.run(fastqc_cmd, check=True)
  print(f"FastQC completed for trimmed reads of {sample}")

except FileNotFoundError as e:
  print(f'Error running command for {sample}: {e}')
  print(f'Skipping sample: {sample}')
except subprocess.CalledProcessError as e:
  print(f'Error running command for {sample}: {e}')
  print(f'Skipping sample: {sample}')

print("Trimmomatic and FastQC processing of trimmed reads completed!")

Processing sample: SLGFSK-T_231336
Running Trimmomatic for SLGFSK-T_231336...
Trimmomatic completed for SLGFSK-T_231336
Running FastQC on trimmed reads for SLGFSK-T_231336...
FastQC completed for trimmed reads of SLGFSK-T_231336
Processing sample: SLGFSK-N_231335
Running Trimmomatic for SLGFSK-N_231335...
Trimmomatic completed for SLGFSK-N_231335
Running FastQC on trimmed reads for SLGFSK-N_231335...
FastQC completed for trimmed reads of SLGFSK-N_231335
Trimmomatic and FastQC processing of trimmed reads completed!


In [ ]:
import os
import subprocess
print("MultiQC processing trimmed Fastqc results...")

subprocess.run(['multiqc', '/content/project/trimmed_reads/Fastqc_results', '-o', '/content/project/trimmed_reads/Fastqc_results'])
print("MultiQC processing complete!")

MultiQC processing trimmed Fastqc results...
MultiQC processing complete!


In [ ]:
# postprocessing reads
# install bwa, samtools, bamtools
import os
print("Installing bioinformatics post processing packages...")

os.system("apt-get update")
os.system("apt-get install -y bwa samtools bamtools")

print('post processing packages installation completed...')
os.system("multiqc --version")
os.system("bwa")
os.system("samtools --version")
os.system("bamtools --version")


Installing bioinformatics post processing packages...
post processing packages installation completed...


0

In [ ]:
# confirming the installation of the packages
!bwa
!samtools --version
!bamtools --version

In [ ]:
import os
import subprocess

# unzip the reference genome
os.system("gunzip /content/project/raw_data/hg19.chr5_12_17.fa.gz")
print('reference file unzip completed...')

print("Indexing the reference genome using bwa...")
# Define the path to the reference genome file
reference_genome = "/content/project/raw_data/hg19.chr5_12_17.fa"

# Run the bwa index command
try:
  subprocess.run(['bwa', 'index', reference_genome], check=True)
  print(f"Indexing of {reference_genome} completed.")
except FileNotFoundError:
  print(f"Error: bwa command not found. Please ensure bwa is installed and in your PATH.")
except subprocess.CalledProcessError as e:
  print(f"Error during bwa index command execution: {e}")

reference file unzip completed...
Indexing the reference genome using bwa...
Indexing of /content/project/raw_data/hg19.chr5_12_17.fa completed.


In [ ]:
# !cp -r "/content/project/Mapping" "/content/drive/MyDrive/project"

In [ ]:
import os
import subprocess

# create Mapping directory
os.makedirs("/content/project/Mapping", exist_ok=True)
print("Mapping directory created")

# get a list of sample names from the trimmed reads directory
trimmed_reads_path = "/content/project/trimmed_reads"
samples = set()
for file_name in os.listdir(trimmed_reads_path):
  if "_r1_paired.fq.gz" in file_name:
    samples.add(file_name.split("_r1_paired.fq.gz")[0])

# define the reference genome path
reference_genome = "/content/project/raw_data/hg19.chr5_12_17.fa"

# process each sample
for sample in samples:
  print(f"Performing BWA alignment for sample: {sample}")

  # define input and output paths
  r1_paired_in = os.path.join(trimmed_reads_path, f"{sample}_r1_paired.fq.gz")
  r2_paired_in = os.path.join(trimmed_reads_path, f"{sample}_r2_paired.fq.gz")
  sam_output = os.path.join("/content/project/Mapping", f"{sample}.sam")

  # construct the BWA command
  # use appropriate RG ID and SM based on sample name
  rg_id = sample.split('_')[1] if '_' in sample else sample
  rg_sm = "Normal" if "N_" in sample else "Tumor"

  bwa_command = [
    'bwa', 'mem', '-R',
    f'@RG\\tID:{rg_id}\\tSM:{rg_sm}',
    reference_genome,
    r1_paired_in,
    r2_paired_in
    ]

  # run the BWA command and redirect output to the SAM file
  try:
    with open(sam_output, 'w') as outfile:
        subprocess.run(bwa_command, check=True, stdout=outfile)
    print(f"BWA alignment completed and saved to {sam_output}")
  except FileNotFoundError as e:
    print(f'Error running BWA for {sample}: {e}')
    print(f'Skipping sample: {sample}')
  except subprocess.CalledProcessError as e:
    print(f'Error running BWA for {sample}: {e}')
    print(f'Skipping sample: {sample}')

print("BWA alignment for all samples completed!")

Mapping directory created
Performing BWA alignment for sample: SLGFSK-N_231335
BWA alignment completed and saved to /content/project/Mapping/SLGFSK-N_231335.sam
Performing BWA alignment for sample: SLGFSK-T_231336
BWA alignment completed and saved to /content/project/Mapping/SLGFSK-T_231336.sam
BWA alignment for all samples completed!


In [ ]:
import os
import subprocess

# mapping directory
trimmed_reads_path = "/content/project/trimmed_reads"
samples = set()
for file_name in os.listdir(trimmed_reads_path):
  if "_r1_paired.fq.gz" in file_name:
    samples.add(file_name.split("_r1_paired.fq.gz")[0])

print("Sample names from trimmed reads:")
for sample in samples:
  print(sample)

def convert_sam_to_sorted_bam_and_index(sample_set):
  sample_list = list(sample_set)

  # process each sample
  for sample in sample_list :
    print(f"Processing sample: {sample}")

    sam_input = os.path.join("/content/project/Mapping", f"{sample}.sam")
    bam_output = os.path.join("/content/project/Mapping", f"{sample}.sorted.bam")

    # check if SAM file exists before processing
    if not os.path.exists(sam_input):
      print(f"Warning: {sam_input} not found, skipping {sample}")
      continue

    # convert SAM to BAM and sort
    print(f"Converting {sample}.sam to sorted BAM...")
    sam_to_bam_command = [
        'samtools', 'view', '-@', '20', '-S', '-b', sam_input
    ]
    sort_bam_command = [
        'samtools', 'sort', '-@', '8', '-o',  bam_output, '-'
    ]
    # sort_bam_command = [
    #     'samtools', 'sort', '-@', '8', '-o',  bam_output, '-' # Use '-' to read from stdin
    # ]

    try:
      # use subprocess.Popen to pipe output from view to sort
      view_process = subprocess.Popen(sam_to_bam_command, stdout=subprocess.PIPE)
      sort_process = subprocess.Popen(sort_bam_command, stdin= view_process.stdout)
      view_process.stdout.close()

      view_process.wait()
      sort_process.wait()

      # check both processes for errors
      if view_process.returncode != 0:
        raise subprocess.CalledProcessError(view_process.returncode, sam_to_bam_command)
      if sort_process.returncode != 0:
        raise subprocess.CalledProcessError(sort_process.returncode, sort_bam_command)

      print(f"Conversion and sorting completed for {sample}.sorted.bam")

      # index BAM file
      print(f"Indexing {sample}.sorted.bam...")
      index_command = [
        'samtools', 'index', bam_output
      ]
      subprocess.run(index_command, check=True)
      print(f"Indexing completed for {sample}.sorted.bam.bai")

    except FileNotFoundError as e:
      print(f'Error: Command not found. Please ensure samtools is installed and in your PATH: {e}')
      print(f'Skipping sample: {sample}')
    except subprocess.CalledProcessError as e:
      print(f'Error during processing for {sample}: {e}')
      print(f'Skipping sample: {sample}')

  print("SAM to sorted BAM conversion and indexing for all samples completed!")

convert_sam_to_sorted_bam_and_index(samples)

Sample names from trimmed reads:
SLGFSK-T_231336
SLGFSK-N_231335
Processing sample: SLGFSK-T_231336
Converting SLGFSK-T_231336.sam to sorted BAM...
Conversion and sorting completed for SLGFSK-T_231336.sorted.bam
Indexing SLGFSK-T_231336.sorted.bam...
Indexing completed for SLGFSK-T_231336.sorted.bam.bai
Processing sample: SLGFSK-N_231335
Converting SLGFSK-N_231335.sam to sorted BAM...
Conversion and sorting completed for SLGFSK-N_231335.sorted.bam
Indexing SLGFSK-N_231335.sorted.bam...
Indexing completed for SLGFSK-N_231335.sorted.bam.bai
SAM to sorted BAM conversion and indexing for all samples completed!


In [ ]:
import os
import subprocess

trimmed_reads_path = "/content/project/trimmed_reads"
samples = set()
for file_name in os.listdir(trimmed_reads_path):
  if "_r1_paired.fq.gz" in file_name:
    samples.add(file_name.split("_r1_paired.fq.gz")[0])

print("Sample names from trimmed reads:")
for sample in samples:
  print(sample)

# BAM file filtering
def bam_file_filtering(sample_set):
  sample_list = list(sample_set)

  for sample in sample_list:
    print(f"Processing sample: {sample}")

    sorted_bam = os.path.join("/content/project/Mapping", f"{sample}.sorted.bam")
    filtered_bam = os.path.join("/content/project/Mapping", f"{sample}.filtered1.bam")

    # Check if sorted BAM file exists before processing
    if not os.path.exists(sorted_bam):
      print(f"Warning: {sorted_bam} not found, skipping {sample}")
      continue

    # Filter BAM file command
    print(f"Filtering {sample}.sorted.bam...")
    bam_filtering_command = [
      'samtools', 'view', '-q', '1', '-f', '0x2', '-F', '0x8', '-b', sorted_bam
    ]

    try:
      # filter BAM file
      with open(filtered_bam, 'wb') as outfile:
        subprocess.run(bam_filtering_command, check=True, stdout=outfile)
      print(f"Filtering completed for {sample}.filtered1.bam")

      # generate flagstat report
      print(f'Generating flagstat for {sample}.filtered1.bam...')
      flagstat_output = os.path.join('/content/project/Mapping', f"{sample}.filtered1.flagstat.txt")

      # flagstat command
      flagstat_command = [
        'samtools', 'flagstat', filtered_bam
      ]

      with open(flagstat_output, 'w') as outfile:
        subprocess.run(flagstat_command, check=True, stdout=outfile)
      print(f"Flagstat completed for {sample}. Results saved to {flagstat_output}")

    except FileNotFoundError as e:
      print(f'Error: Command not found. Please ensure samtools is installed and in your PATH: {e}')
      print(f'Skipping sample: {sample}')
    except subprocess.CalledProcessError as e:
      print(f'Error during processing for {sample}: {e}')

  print("BAM file filtering for all samples completed!")

bam_file_filtering(samples)

Sample names from trimmed reads:
SLGFSK-T_231336
SLGFSK-N_231335
Processing sample: SLGFSK-T_231336
Filtering SLGFSK-T_231336.sorted.bam...
Filtering completed for SLGFSK-T_231336.filtered1.bam
Generating flagstat for SLGFSK-T_231336.filtered1.bam...
Flagstat completed for SLGFSK-T_231336. Results saved to /content/project/Mapping/SLGFSK-T_231336.filtered1.flagstat.txt
Processing sample: SLGFSK-N_231335
Filtering SLGFSK-N_231335.sorted.bam...
Filtering completed for SLGFSK-N_231335.filtered1.bam
Generating flagstat for SLGFSK-N_231335.filtered1.bam...
Flagstat completed for SLGFSK-N_231335. Results saved to /content/project/Mapping/SLGFSK-N_231335.filtered1.flagstat.txt
BAM file filtering for all samples completed!


In [ ]:
import os
import subprocess

# get sample names from trimmed reads directory
trimmed_reads_path = "/content/project/trimmed_reads"
mapping_dir = "/content/project/Mapping"
samples = set()
for file_name in os.listdir(trimmed_reads_path):
  if "_r1_paired.fq.gz" in file_name:
    samples.add(file_name.split("_r1_paired.fq.gz")[0])

print("Sample names from trimmed reads:")
for sample in samples:
  print(sample)

def mark_duplicates(sample_set, cleanup_intermediates=True):
  sample_list = list(sample_set)

  # process each sample for duplicate marking
  for sample in sample_list:
    print(f"Processing sample: {sample}")

    # define file paths
    filtered_bam = os.path.join(mapping_dir, f"{sample}.filtered1.bam")
    namecollate_prefix = os.path.join(mapping_dir, f"{sample}.namecollate")
    namecollate_bam = f"{namecollate_prefix}.bam"
    fixmate_bam = os.path.join(mapping_dir, f"{sample}.fixmate.bam")
    positionsort_bam = os.path.join(mapping_dir, f"{sample}.positionsort.bam")
    clean_bam = os.path.join(mapping_dir, f"{sample}.clean.bam")

    # check if filtered BAM file exists before processing
    if not os.path.exists(filtered_bam):
      print(f"Warning: {filtered_bam} not found, skipping {sample}")
      continue

    print(f"Starting duplicate marking for {sample}...")

    try:
        # step 1: samtools collate 
        print(f"Running samtools collate for {sample}...")
        collate_command = ['samtools', 'collate', filtered_bam, namecollate_prefix]
        subprocess.run(collate_command, check=True)
        print(f"samtools collate completed for {sample}")

        # step 2: samtools fixmate
        print(f"Running samtools fixmate for {sample}...")
        fixmate_command = ['samtools', 'fixmate', '-m', '-O', 'BAM', namecollate_bam, fixmate_bam]
        result = subprocess.run(fixmate_command, check=True, capture_output=True, text=True)
        print(f"samtools fixmate completed for {sample}")

        # step 3: samtools sort by position
        print(f"Running samtools sort for {sample}...")
        sort_command = ['samtools', 'sort', '-@', '4', '-o', positionsort_bam, fixmate_bam]
        subprocess.run(sort_command, check=True)
        print(f"samtools sort completed for {sample}")

        # step 4: samtools markdup
        print(f"Running samtools markdup for {sample}...")
        markdup_command = ['samtools', 'markdup', '-@', '4', '-r', positionsort_bam, clean_bam]
        subprocess.run(markdup_command, check=True)
        print(f"samtools markdup completed for {sample}")

        # step 5: Index the clean BAM file
        print(f"Indexing {sample}.clean.bam...")
        index_command = ['samtools', 'index', clean_bam]
        subprocess.run(index_command, check=True)
        print(f"Indexing completed for {sample}.clean.bam")

        # optional clean up of intermediate files for storage management
        if cleanup_intermediates:
          intermediate_files = [namecollate_bam, fixmate_bam, positionsort_bam]
          for file_path in intermediate_files:
            if os.path.exists(file_path):
              os.remove(file_path)
              print(f"Cleaned up: {os.path.basename(file_path)}")

        # generate final statistics
        print(f"Generating final flagstat for {sample}.clean.bam...")
        flagstat_output = os.path.join(mapping_dir, f"{sample}.clean.flagstat.txt")
        flagstat_command = ['samtools', 'flagstat', clean_bam]

        with open(flagstat_output, 'w') as outfile:
          subprocess.run(flagstat_command, check=True, stdout=outfile)
        print(f"Final flagstat saved to {flagstat_output}")

    except FileNotFoundError as e:
      print(f'Error: Command not found. Please ensure samtools is installed: {e}')
      print(f'Skipping sample: {sample}')
      continue
    except subprocess.CalledProcessError as e:
      print(f'Error during processing for {sample}: {e}')
      print(f'Command failed with return code: {e.returncode}')
      print(f'Skipping sample: {sample}')
      continue

  print("Duplicate marking for all samples completed!")

mark_duplicates(samples)

In [ ]:
!sudo apt-get install freebayes

In [ ]:
import os
import subprocess

# Get sample names from trimmed reads directory
trimmed_reads_path = "/content/project/trimmed_reads"
mapping_dir = "/content/project/Mapping"
reference_genome = "/content/project/hg19.chr5_12_17.fa"

samples = set()
for file_name in os.listdir(trimmed_reads_path):
    if "_r1_paired.fq.gz" in file_name:
        samples.add(file_name.split("_r1_paired.fq.gz")[0])

print("Sample names from trimmed reads:")
for sample in samples:
    print(sample)

def bam_left_align(sample_set):
    sample_list = list(sample_set)

    # Process each sample for left alignment
    for sample in sample_list:
        print(f"Processing sample: {sample}")

        # Define file paths
        clean_bam = os.path.join(mapping_dir, f"{sample}.clean.bam")
        leftalign_bam = os.path.join(mapping_dir, f"{sample}.leftAlign.bam")

        # Check if clean BAM file exists before processing
        if not os.path.exists(clean_bam):
            print(f"Warning: {clean_bam} not found, skipping {sample}")
            continue

        # # Check if reference FASTA exists
        # if not os.path.exists(reference_fasta):
        #     print(f"Warning: Reference FASTA {reference_fasta} not found, skipping {sample}")
        #     continue

        print(f"Starting left alignment for {sample}...")

        try:
            # BAM left align command: cat clean.bam | bamleftalign -f ref.fa -m 5 -c > leftAlign.bam
            print(f"Running bamleftalign for {sample}...")

            # Using subprocess.Popen to handle the pipe operation
            cat_process = subprocess.Popen(['cat', clean_bam], stdout=subprocess.PIPE)

            bamleftalign_command = [
                'bamleftalign',
                '-f', reference_genome,
                '-m', '5',
                '-c'
            ]

            with open(leftalign_bam, 'w') as outfile:
                bamleftalign_process = subprocess.Popen(
                    bamleftalign_command,
                    stdin=cat_process.stdout,
                    stdout=outfile,
                    stderr=subprocess.PIPE
                )

                # Close cat stdout to allow it to receive SIGPIPE
                cat_process.stdout.close()

                # Wait for both processes to complete
                cat_returncode = cat_process.wait()
                bamleftalign_returncode = bamleftalign_process.wait()

            print(f"Left alignment completed for {sample}")

        except FileNotFoundError as e:
            print(f'Error: Command not found. Please ensure bamleftalign is installed: {e}')
            print(f'Skipping sample: {sample}')
            continue
        except subprocess.CalledProcessError as e:
            print(f'Error during processing for {sample}: {e}')
            print(f'Skipping sample: {sample}')
            continue
        except Exception as e:
            print(f'Unexpected error for {sample}: {e}')
            print(f'Skipping sample: {sample}')
            continue

    print("BAM left alignment for all samples completed!")

# Call the function with the samples set
bam_left_align(samples)

In [ ]:
import os
import subprocess

# Get sample names from trimmed reads directory
trimmed_reads_path = "/content/project/trimmed_reads"
mapping_dir = "/content/project/Mapping"
reference_genome = "/content/project/raw_data/hg19.chr5_12_17.fa"

samples = set()
for file_name in os.listdir(trimmed_reads_path):
    if "_r1_paired.fq.gz" in file_name:
        samples.add(file_name.split("_r1_paired.fq.gz")[0])

print("Sample names from trimmed reads:")
for sample in samples:
    print(sample)

def bam_recalibration_and_refiltering(sample_set):
    sample_list = list(sample_set)

    for sample in sample_list:
        print(f"Processing sample: {sample}")

        # Define file paths
        leftalign_bam = os.path.join(mapping_dir, f"{sample}.leftAlign.bam")
        recalibrate_bam = os.path.join(mapping_dir, f"{sample}.recalibrate.bam")
        refilter_bam = os.path.join(mapping_dir, f"{sample}.refilter.bam")

        # Check if leftAlign BAM file exists
        if not os.path.exists(leftalign_bam):
            print(f"Warning: {leftalign_bam} not found, skipping {sample}")
            continue

        # Check if reference genome exists
        if not os.path.exists(reference_genome):
            print(f"Warning: {reference_genome} not found, skipping {sample}")
            continue

        try:
            # Step 1: Recalibration - samtools calmd -@ 32 -b leftAlign.bam reference.fa > recalibrate.bam
            print(f"Running samtools calmd for {sample}...")
            calmd_command = [
                'samtools', 'calmd',
                '-@', '32',
                '-b',
                leftalign_bam,
                reference_genome
            ]

            with open(recalibrate_bam, 'w') as outfile:
                subprocess.run(calmd_command, check=True, stdout=outfile)
            print(f"Recalibration completed for {sample}")

            # Step 2: Refiltering - bamtools filter -in recalibrate.bam -mapQuality "<=254" > refilter.bam
            print(f"Running bamtools filter for {sample}...")
            filter_command = [
                'bamtools', 'filter',
                '-in', recalibrate_bam,
                '-mapQuality', '<=254'
            ]

            with open(refilter_bam, 'w') as outfile:
                subprocess.run(filter_command, check=True, stdout=outfile)
            print(f"Refiltering completed for {sample}")

        except FileNotFoundError as e:
            print(f'Error: Command not found: {e}')
            print(f'Skipping sample: {sample}')
            continue
        except subprocess.CalledProcessError as e:
            print(f'Error during processing for {sample}: {e}')
            print(f'Skipping sample: {sample}')
            continue

    print("BAM recalibration and refiltering for all samples completed!")
bam_recalibration_and_refiltering(samples)

In [ ]:
#getting variant calling package
!wget https://sourceforge.net/projects/varscan/files/VarScan.v2.3.9.jar

--2025-08-30 22:28:19--  https://sourceforge.net/projects/varscan/files/VarScan.v2.3.9.jar
Resolving sourceforge.net (sourceforge.net)... 104.18.12.149, 104.18.13.149, 2606:4700::6812:d95, ...
Connecting to sourceforge.net (sourceforge.net)|104.18.12.149|:443... connected.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: https://sourceforge.net/projects/varscan/files/VarScan.v2.3.9.jar/ [following]
--2025-08-30 22:28:20--  https://sourceforge.net/projects/varscan/files/VarScan.v2.3.9.jar/
Reusing existing connection to sourceforge.net:443.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: https://sourceforge.net/projects/varscan/files/VarScan.v2.3.9.jar/download [following]
--2025-08-30 22:28:20--  https://sourceforge.net/projects/varscan/files/VarScan.v2.3.9.jar/download
Reusing existing connection to sourceforge.net:443.
HTTP request sent, awaiting response... 302 Found
Location: https://downloads.sourceforge.net/project/varscan/VarScan.v2

In [ ]:
import os
import subprocess

# create the variant dir
os.makedirs("/content/project/Variants", exist_ok=True)


# Get sample names from trimmed reads directory
trimmed_reads_path = "/content/project/trimmed_reads"
mapping_dir = "/content/project/Mapping"
variants_dir = "/content/project/Variants"
reference_genome = "/content/project/raw_data/hg19.chr5_12_17.fa"

samples = set()
for file_name in os.listdir(trimmed_reads_path):
    if "_r1_paired.fq.gz" in file_name:
        samples.add(file_name.split("_r1_paired.fq.gz")[0])

print("Sample names from trimmed reads:")
for sample in samples:
    print(sample)

def samtools_mpileup(sample_set):
    """
    Generates mpileup files using samtools mpileup.
    """
    sample_list = list(sample_set)

    # Ensure variants directory exists
    os.makedirs(variants_dir, exist_ok=True)

    for sample in sample_list:
        print(f"Processing sample: {sample}")

        # Define file paths
        refilter_bam = os.path.join(mapping_dir, f"{sample}.refilter.bam")
        pileup_file = os.path.join(variants_dir, f"{sample}.pileup")

        # Check if refilter BAM file exists
        if not os.path.exists(refilter_bam):
            print(f"Warning: {refilter_bam} not found, skipping {sample}")
            continue

        # Check if reference genome exists
        if not os.path.exists(reference_genome):
            print(f"Warning: {reference_genome} not found, skipping {sample}")
            continue

        print(f"Running samtools mpileup for {sample}...")

        try:
            # samtools mpileup -f reference.fa refilter.bam --min-MQ 1 --min-BQ 28 > pileup
            mpileup_command = [
                'samtools', 'mpileup',
                '-f', reference_genome,
                refilter_bam,
                '--min-MQ', '1',
                '--min-BQ', '28'
            ]

            with open(pileup_file, 'w') as outfile:
                subprocess.run(mpileup_command, check=True, stdout=outfile)

            print(f"Mpileup completed for {sample}")

        except FileNotFoundError as e:
            print(f'Error: samtools not found: {e}')
            print(f'Skipping sample: {sample}')
            continue
        except subprocess.CalledProcessError as e:
            print(f'Error during mpileup for {sample}: {e}')
            print(f'Skipping sample: {sample}')
            continue

    print("Samtools mpileup for all samples completed!")

# Call the function
samtools_mpileup(samples)

Sample names from trimmed reads:
SLGFSK-T_231336
SLGFSK-N_231335
Processing sample: SLGFSK-T_231336
Running samtools mpileup for SLGFSK-T_231336...
Mpileup completed for SLGFSK-T_231336
Processing sample: SLGFSK-N_231335
Running samtools mpileup for SLGFSK-N_231335...
Mpileup completed for SLGFSK-N_231335
Samtools mpileup for all samples completed!


In [ ]:
import os
import subprocess

def varscan_somatic_calling(samples):
    """Run VarScan somatic calling for paired normal/tumor samples"""

    # Process each sample pair
    for normal_sample, tumor_sample in samples:
        print(f"Processing sample: {normal_sample} (normal) vs {tumor_sample} (tumor)...")

        # Define file paths (inside the loop)
        normal_pileup = os.path.join('/content/project/Variants', f"{normal_sample}.pileup")
        tumor_pileup = os.path.join('/content/project/Variants', f"{tumor_sample}.pileup")

        # Create output/result prefix using sample identifiers
        sample_id = normal_sample.split('-')[0]
        result = os.path.join('/content/project/Variants', f"{sample_id}_somatic")

        # Purity variables
        normal_purity = 1
        tumor_purity = 0.5

        # Check if input files exist
        if not os.path.exists(normal_pileup):
            print(f"Missing file: {normal_pileup}")
            continue
        if not os.path.exists(tumor_pileup):
            print(f"Missing file: {tumor_pileup}")
            continue

        print(f"Running VarScan: {normal_sample} vs {tumor_sample}")

        # VarScan command
        varscan_command = [
            'java', '-jar', 'VarScan.v2.3.9.jar', 'somatic',
            normal_pileup, tumor_pileup, result,
            '--normal-purity', str(normal_purity),
            '--tumor-purity', str(tumor_purity),
            '--output-vcf', '1'
        ]

        try:
            # Run the command
            subprocess.run(varscan_command, check=True)
            print(f"✓ VarScan completed for {normal_sample} vs {tumor_sample}")

        except subprocess.CalledProcessError as e:
            print(f"✗ Error during VarScan for {normal_sample} vs {tumor_sample}: {e}")
            continue

    print("VarScan somatic calling completed for all samples!")

# Define sample pairs and run
samples = [
    ("SLGFSK-N_231335", "SLGFSK-T_231336")
]

varscan_somatic_calling(samples)

Processing sample: SLGFSK-N_231335 (normal) vs SLGFSK-T_231336 (tumor)...
Running VarScan: SLGFSK-N_231335 vs SLGFSK-T_231336
✓ VarScan completed for SLGFSK-N_231335 vs SLGFSK-T_231336
VarScan somatic calling completed for all samples!


In [ ]:
# !cp -r "/content/project/Variants" "/content/drive/MyDrive/project"

In [ ]:
print("Installing bgzip, tabix, and bcftools..")
import os
# !sudo apt-get update
# !sudo apt-get install -y bcftools
# !sudo apt-get install -y bgzip tabix
os.system("sudo apt-get install tabix")
# # !sudo apt-get install bcftools

# os.system("sudo apt-get update")
# os.system("apt-get install -y htslib-tools")
# # os.system("sudo apt-get install -y htslib-tools")  # Contains bgzip and tabix
# os.system("sudo apt-get install -y bcftools")     # For bcftools merge


print("Installation complete!")

Installing bgzip, tabix, and bcftools..
Installation complete!


In [ ]:
!cp -r "/content/project/Variants" "/content/drive/MyDrive/project"

In [ ]:
import os
import subprocess

def merge_vcf_files(sample):
  """Merge VarScan VCF files for paired normal/tumor samples"""
  variants_dir = "/content/project/Variants"

  # Process each sample pair

  # define file paths
  snp_vcf = os.path.join(variants_dir, f"{sample}.snp.vcf")
  indel_vcf = os.path.join(variants_dir, f"{sample}.indel.vcf")
  snp_vcf_gz = os.path.join(variants_dir, f"{sample}.snp.vcf.gz")
  indel_vcf_gz = os.path.join(variants_dir, f"{sample}.indel.vcf.gz")
  merged_vcf = os.path.join(variants_dir, f"{sample}_merged.vcf")

  print(f"Starting VCF merge process for {sample}...")
  # Check if input files exist
  if not os.path.exists(snp_vcf):
    print(f"Missing file: {snp_vcf}")
    return False
  if not os.path.exists(indel_vcf):
    print(f"Missing file: {indel_vcf}")
    return False
  try:
    # Step 1: compress SNP VCF file
    print(f"Compressing SNP VCF file...")
    bgzip_snp_command = ['bgzip', '-c', snp_vcf]
    with open(snp_vcf_gz, 'wb') as outfile:
      subprocess.run(bgzip_snp_command, check=True, stdout=outfile)
    print(f" Created: {os.path.basename(snp_vcf_gz)}")

    # Step 2: Compress indel VCF file
    print(f"Compressing indel VCF file...")
    bgzip_indel_command = ['bgzip', '-c', indel_vcf]
    with open(indel_vcf_gz, 'wb') as outfile:
      subprocess.run(bgzip_indel_command, stdout=outfile, check=True)
    print(f" Created: {os.path.basename(indel_vcf_gz)}")

    # Step 3: Index SNP VCF && indel VCF file
    print(f"Indexing SNP VCF file...")
    # tabix_snp_command = ['tabix', snp_vcf_gz]
    tabix_snp_command = ['tabix', '-p', 'vcf', snp_vcf_gz]
    subprocess.run(tabix_snp_command, check=True)
    print(f" Indexed: {os.path.basename(snp_vcf_gz)}")

    print(f"Indexing indel VCF file...")
    # tabix_indel_command = ['tabix', indel_vcf_gz]
    tabix_indel_command = ['tabix', '-p', 'vcf', indel_vcf_gz]
    subprocess.run(tabix_indel_command, check=True)
    print(f" Indexed: {os.path.basename(indel_vcf_gz)}")

    # Step 4: Merge VCF files
    print(f"Merging VCF files...")
    merge_vcf_command = ['bcftools', 'merge', '--force-samples', snp_vcf_gz, indel_vcf_gz]
    # merge_vcf_command = ['bcftools', 'merge', snp_vcf_gz, indel_vcf_gz]
    with open(merged_vcf, 'w') as outfile:
      subprocess.run(merge_vcf_command, stdout=outfile, check=True)
    print(f" Created merged VCF: {os.path.basename(merged_vcf)}")

    return True
  except subprocess.CalledProcessError as e:
    print(f"Error during VCF merge for {sample}: {e}")
    return False
  except FileNotFoundError as e:
    print(f"Error: Command not found: {e}")
    return False

merge_vcf_files("SLGFSK")

Starting VCF merge process for SLGFSK...
Compressing SNP VCF file...
 Created: SLGFSK.snp.vcf.gz
Compressing indel VCF file...
 Created: SLGFSK.indel.vcf.gz
Indexing SNP VCF file...
 Indexed: SLGFSK.snp.vcf.gz
Indexing indel VCF file...
 Indexed: SLGFSK.indel.vcf.gz
Merging VCF files...
 Created merged VCF: SLGFSK_merged.vcf


True

In [ ]:
!rm -r /content/snpEff
# !rm -r /content/snpEff_latest_core.zip.1

In [ ]:
import os
import subprocess

def download_and_prepare_snpeff():
  # print("Downloading snpEff...")
  # subprocess.run(
  #     [ "wget", "-O", "snpEff_latest_core.zip", "https://snpeff.odsp.astrazeneca.com/versions/snpEff_latest_core.zip"
  # ], check=True)
  #     # "wget", "-O", "snpEff_latest_core.zip", "https://sourceforge.net/projects/snpeff/files/snpEff_latest_core.zip/download",
  # # ], check=True)
  # print("✓ Download complete")

  # print("Unzipping snpEff...")
  # subprocess.run(["unzip", "snpEff_latest_core.zip"], check=True)
  # print("snpEff unzipped successfully!")

  print("Downloading snpEff database...")
  subprocess.run([
      "java", "-Xmx8g", "-jar", "snpEff.jar", "download", "hg19"
  ], check=True)

  print("Annotating variants with snpEff...")
  merged_vcf = "/content/project/Variants/SLGFSK_merged.vcf"
  annotated_vcf = "/content/project/Variants/SLGFSK_merged_annotated.vcf"

  # with open(merged_vcf, "r") as infile:
  with open(annotated_vcf, "wb") as outfile:
    subprocess.run([
        "java", "-Xmx8g", "-jar", "snpEff.jar", "hg19", merged_vcf
        ], stdout=outfile, check=True)

download_and_prepare_snpeff()

In [ ]:
def download_and_prepare_snpeff():
  print("Downloading snpEff...")
  subprocess.run(
      [ "wget", "-O", "snpEff_latest_core.zip", "https://sourceforge.net/projects/snpeff/files/snpEff_latest_core.zip"
  ], check=True)
  print("Download complete")

  print("Unzipping snpEff...")
  subprocess.run(["unzip", "snpEff_latest_core.zip"], check=True)
  print(" snpEff unzipped successfully!")

  # update java to Version-21 for compatibility
  # !apt-get update
  # !apt-get install -y openjdk-21-jdk
  subprocess.run(["apt-get", "update"], check=True)
  subprocess.run(["apt-get", "install", "-y", "openjdk-21-jdk"], check=True)

  # print("Downloading snpEff database...")

  # change to snpEff directory
  if os.path.exists("snpEff"):
    os.chdir("snpEff")
    print("Changed to snpEff directory")

  # download snpEff hg19 database
  print("Downloading snpEff database...")
  subprocess.run([
      "java", "-Xmx8g", "-jar", "snpEff.jar", "download", "hg19"
  ], check=True)
  print("Database download successful")

download_and_prepare_snpeff()

In [ ]:
import subprocess
import os

def run_snpeff_annotation():
  # define file paths
  merged_vcf = "/content/project/Variants/SLGFSK_merged.vcf"
  annotated_vcf = "/content/project/Variants/SLGFSK_merged_annotated.vcf"

  # change to snpEff directory
  if os.path.exists("snpEff"):
    os.chdir("snpEff")
    print("Changed to snpEff directory")

  # download database with error capture
  print("Downloading snpEff database...")
  subprocess.run([
      "java", "-Xmx8g", "-jar", "snpEff.jar", "download", "hg19"
  ], capture_output=True, text=True, check=True)
  print("Database download successful")

  print("Annotating variants with snpEff...")
  try:
      with open(annotated_vcf, "w") as outfile:
          result = subprocess.run([
              "java", "-Xmx8g", "-jar", "snpEff.jar", "hg19", merged_vcf
          ], stdout=outfile, stderr=subprocess.PIPE, text=True, check=True)
      print("Annotation complete!")
      print(f"Output saved to: {annotated_vcf}")
  except subprocess.CalledProcessError as e:
      print(f"Annotation failed with exit code: {e.returncode}")
      print(f"Error output: {e.stderr}")

run_snpeff_annotation()

Java version check passed
Database download successful
Annotating variants with snpEff...
Annotation complete!
Output saved to: /content/project/Variants/SLGFSK_merged_annotated.vcf


In [ ]:
#clinical annotation using gemini
wget https://raw.github.com/arq5x/gemini/master/gemini/scripts/gemini_install.py
#python gemini_install.py /usr/local /usr/local/share/gemini

gemini load -v Variants/SLGFSK.ann.vcf -t snpEff Annotation/gemini.db

In [5]:
import subprocess
subprocess.run(["wget", "https://raw.github.com/arq5x/gemini/master/gemini/scripts/gemini_install.py"], check=True)
print("gemini successfully installed!")

gemini successfully installed!


In [6]:
!gemini --version

0.3.4


In [7]:
import os
import subprocess

def load_gemini():
  # create annotation directory for gemini
  os.makedirs("/content/project/Annotation", exist_ok=True)

  # define file paths
  ann_vcf_file="/content/project/Variants/SLGFSK_merged_annotated.vcf"
  gemini_db="/content/project/Annotation/gemini.db"

  try:
    subprocess.run(["gemini", "load", "-v", ann_vcf_file, "-t", "snpEff", gemini_db], check=True)
    print("Gemini database loaded successfully!")
  except subprocess.CalledProcessError as e:
    print(f"Error loading Gemini database: {e}")

load_gemini()

Gemini database loaded successfully!
